# Face-shape classifier: fine-tuning MobileNetV2

Trains an image model to classify face shape (heart, oblong, oval, round, square)
and measures its accuracy **honestly**:

1. Every photo is cropped to the face the same way the app will crop webcam frames
   (MediaPipe face landmarks → level the eyes → square crop with margin → 224×224, never stretched).
2. Photos are grouped **by person** using face-recognition embeddings, and no person appears
   in both training and test. The dataset has the same celebrities across its own splits, so
   without this a model can score well by recognising people instead of face shapes.
3. MobileNetV2 (pretrained on ImageNet) gets a new 5-way output layer, trained first with the
   pretrained part frozen, then fine-tuned end to end at a small learning rate.
4. Accuracy, per-shape results and a confusion matrix on the person-disjoint test set.
5. Export to `.keras` and `.tflite` for the browser.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put `image calibrating 3.zip`
in Google Drive under `MyDrive/glasses-tryon/`. Then Runtime → Run all.

In [ ]:
# Settings
DRIVE_DIR = '/content/drive/MyDrive/glasses-tryon'
DATA_ZIP = f'{DRIVE_DIR}/image calibrating 3.zip'
BACKBONE = 'MobileNetV2'          # or 'EfficientNetB0' for a bigger, often more accurate model
IMG_SIZE = 224
CROP_MARGIN = 1.35                # crop side = this × the face's larger dimension
SAME_PERSON_SIMILARITY = 0.70     # Facenet512 cosine similarity a group's photos must average to count as one person
MAX_RANDOM_PAIR_SIMILARITY = 0.4  # stop if two random photos are typically more alike than this — the embeddings have gone wrong
TEST_SHARE_FOLDS = 5              # 1 of 5 person-disjoint folds is held out for test (~20%)
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 25
FINE_TUNE_LAYERS = 60             # how many of the backbone's top layers to unfreeze
SEED = 42

In [ ]:
!pip -q install mediapipe deepface

In [ ]:
import json, os, random, shutil, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
import keras
from keras import layers

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path(DRIVE_DIR) / time.strftime('run-%Y%m%d-%H%M')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Results will be saved to', RUN_DIR)

## 1. Unzip the dataset and list every photo
All of Roboflow's train/valid/test folders are pooled; the person-level split below replaces them.

In [ ]:
DATA = Path('/content/data')
if not DATA.exists():
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA)

SHAPES = ['heart', 'oblong', 'oval', 'round', 'square']
photos = []
for p in sorted(DATA.rglob('*')):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'} and p.parent.name.lower() in SHAPES:
        photos.append({'path': str(p), 'label': p.parent.name.lower(), 'roboflow_split': p.parent.parent.name})
print(len(photos), 'photos')
print({s: sum(ph['label'] == s for ph in photos) for s in SHAPES})

## 2. Crop every face the way the app will
Same MediaPipe face-landmark model the app uses. The image is rotated so the outer eye corners
are level, then a square around the landmarks is cut out (padding with black past the edges) and
resized to 224×224. Nothing is stretched, so face proportions are preserved.

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

MODEL_PATH = '/content/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
        MODEL_PATH)
landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))

EYE_LEFT_OUTER, EYE_RIGHT_OUTER = 33, 263

def crop_face(path):
    """Returns a 224×224 RGB face crop, or None when no face is found."""
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    w, h = img.size
    result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(img)))
    if not result.face_landmarks:
        return None
    pts = np.array([[lm.x * w, lm.y * h] for lm in result.face_landmarks[0]])
    a, b = pts[EYE_LEFT_OUTER], pts[EYE_RIGHT_OUTER]
    theta = np.arctan2(b[1] - a[1], b[0] - a[0])
    if abs(theta) > np.pi / 2:
        theta -= np.sign(theta) * np.pi
    center = pts.mean(axis=0)
    # PIL rotates counter-clockwise for positive angles; this levels the eyes.
    img = img.rotate(np.degrees(theta), resample=Image.BICUBIC, center=tuple(center))
    c, s = np.cos(theta), np.sin(theta)
    rotated = (pts - center) @ np.array([[c, -s], [s, c]]) + center
    lo, hi = rotated.min(axis=0), rotated.max(axis=0)
    mid = (lo + hi) / 2
    half = CROP_MARGIN * max(hi - lo) / 2
    box = tuple(int(round(v)) for v in (mid[0] - half, mid[1] - half, mid[0] + half, mid[1] + half))
    return img.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)

CROPS = Path('/content/crops')
CROPS.mkdir(exist_ok=True)
kept, no_face = [], 0
for i, ph in enumerate(photos):
    out = CROPS / f'{i:05d}.png'
    if not out.exists():
        face = crop_face(ph['path'])
        if face is None:
            no_face += 1
            continue
        face.save(out)
    kept.append({**ph, 'crop': str(out)})
    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(photos)} cropped')
photos = kept
print(len(photos), 'faces kept,', no_face, 'photos with no face found')

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, ph in zip(axes.flat, random.sample(photos, 16)):
    ax.imshow(Image.open(ph['crop'])); ax.set_title(ph['label']); ax.axis('off')
plt.show()

## 3. Group photos by person
Facenet512 face embeddings (via DeepFace) on a tight crop of each face. Photos are grouped with
average-linkage clustering: a group only forms when its photos are, **on average**, at least
`SAME_PERSON_SIMILARITY` alike. (Joining any two similar photos instead lets chains of
look-alikes merge thousands of different people into one "person".)
The grid shows the largest groups so you can check each row really is one person. This dataset
repeats some celebrities 70–100+ times, each with a single label, which is exactly why the split
has to be by person: otherwise a model can score well by recognising who someone is.

In [ ]:
from deepface import DeepFace
from sklearn.cluster import AgglomerativeClustering

# The training crops include hair and background; face recognition works best on the face itself.
face_px = int(round(IMG_SIZE / CROP_MARGIN))
edge = (IMG_SIZE - face_px) // 2
embeddings = []
for i, ph in enumerate(photos):
    face = np.asarray(Image.open(ph['crop']))[edge:edge + face_px, edge:edge + face_px]
    rep = DeepFace.represent(img_path=face[:, :, ::-1].copy(), model_name='Facenet512',
                             detector_backend='skip', enforce_detection=False)
    embeddings.append(rep[0]['embedding'])
    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(photos)} embedded')
E = np.asarray(embeddings, dtype=np.float64)
E /= np.linalg.norm(E, axis=1, keepdims=True)

# Sanity check: two random photos are almost always different people, so they should look unalike.
rng = np.random.default_rng(SEED)
a, b = rng.integers(0, len(E), 20000), rng.integers(0, len(E), 20000)
random_pairs = (E[a] * E[b]).sum(axis=1)
print('Random-pair similarity, percentiles 50/90/99:', np.percentile(random_pairs, [50, 90, 99]).round(3))
if np.median(random_pairs) > MAX_RANDOM_PAIR_SIMILARITY:
    raise RuntimeError('Random photos look too alike, so the face embeddings are not telling people apart. '
                       'Check the face crops in section 2.')

distance = np.clip(1 - E @ E.T, 0, 2)
groups = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average',
                                 distance_threshold=1 - SAME_PERSON_SIMILARITY).fit_predict(distance)

sizes = np.bincount(groups)
multi = [g for g in range(len(sizes)) if sizes[g] > 1]
mixed = [g for g in multi if len({photos[i]['label'] for i in np.nonzero(groups == g)[0]}) > 1]
print(f'{len(photos)} photos → {len(sizes)} people; {len(multi)} people have 2+ photos; '
      f'largest group {sizes.max()} photos')
print(f'{len(mixed)} people have photos labeled with different shapes '
      f'({sum(sizes[g] for g in mixed)} photos) — a direct measure of label disagreement')
print('Largest groups:', sorted(sizes.tolist())[-10:], '— the dataset repeats some celebrities dozens of times')

show = sorted(multi, key=lambda g: -sizes[g])[:6]
if show:
    fig, axes = plt.subplots(len(show), 6, figsize=(12, 2.2 * len(show)))
    for row, g in zip(np.atleast_2d(axes), show):
        for ax in row:
            ax.axis('off')
        for ax, i in zip(row, np.nonzero(groups == g)[0][:6]):
            ax.imshow(Image.open(photos[i]['crop'])); ax.set_title(photos[i]['label'], fontsize=9)
    plt.tight_layout(); plt.show()

## 4. Person-disjoint split
Stratified by shape, grouped by person: ~20% test, then ~12% of the rest for validation
(used only to decide when to stop training). Nobody appears in more than one split.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

labels = np.array([SHAPES.index(ph['label']) for ph in photos])
idx = np.arange(len(photos))
outer = StratifiedGroupKFold(n_splits=TEST_SHARE_FOLDS, shuffle=True, random_state=SEED)
rest, test_idx = next(outer.split(idx, labels, groups))
inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SEED)
tr, va = next(inner.split(rest, labels[rest], groups[rest]))
train_idx, val_idx = rest[tr], rest[va]

assert not set(groups[train_idx]) & set(groups[test_idx])
assert not set(groups[val_idx]) & set(groups[test_idx])
if len(test_idx) < 0.1 * len(photos):
    raise RuntimeError(f'The test set has only {len(test_idx)} photos; check the person groups in section 3.')
for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:5s} {len(ids):5d} photos of {len(set(groups[ids])):4d} people', np.bincount(labels[ids], minlength=5))

## 5. Build and train the model
Augmentation that keeps face shape intact: horizontal flips, small rotations, brightness and
contrast. No zooming or stretching, which would turn one face shape into another.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
paths = np.array([ph['crop'] for ph in photos])

def dataset(ids, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths[ids], labels[ids]))
    if shuffle:
        ds = ds.shuffle(len(ids), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (tf.cast(tf.io.decode_png(tf.io.read_file(p), channels=3), tf.float32), y),
                num_parallel_calls=AUTOTUNE)
    return ds.batch(32).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = dataset(train_idx, True), dataset(val_idx, False), dataset(test_idx, False)

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(8 / 360, fill_mode='constant'),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
], name='augment')

if BACKBONE == 'EfficientNetB0':
    base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    normalize = layers.Identity()                      # EfficientNet takes 0–255 input
else:
    base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    normalize = layers.Rescaling(1 / 127.5, offset=-1)  # MobileNetV2 expects −1…1
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = normalize(x)
x = base(x, training=False)   # keeps BatchNorm in inference mode, even while fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(SHAPES), activation='softmax')(x)
model = keras.Model(inputs, outputs)

from sklearn.utils.class_weight import compute_class_weight
class_weight = dict(enumerate(compute_class_weight('balanced', classes=np.arange(5), y=labels[train_idx])))

# Stage 1: train only the new output layer.
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weight)

# Stage 2: unfreeze the top of the backbone and fine-tune gently.
base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weight, callbacks=[stop])

acc = h1.history['accuracy'] + h2.history['accuracy']
val = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, label='train'); plt.plot(val, label='validation')
plt.axvline(len(h1.history['accuracy']) - 0.5, color='gray', ls='--', label='fine-tuning starts')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.show()

## 6. Accuracy on the person-disjoint test set
Each test photo is classified as-is and mirrored, and the two predictions are averaged.
These photos (and everyone in them) were never seen during training or early stopping.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

probs = []
for x, _ in test_ds:
    probs.append((model(x, training=False) + model(tf.image.flip_left_right(x), training=False)).numpy() / 2)
pred = np.concatenate(probs).argmax(axis=1)
truth = labels[test_idx]
accuracy = float((pred == truth).mean())

print(f'Test accuracy: {accuracy:.1%} on {len(test_idx)} photos of {len(set(groups[test_idx]))} people '
      f'never seen in training (chance: 20%)\n')
print(classification_report(truth, pred, labels=range(len(SHAPES)), target_names=SHAPES, digits=3, zero_division=0))
cm = confusion_matrix(truth, pred, labels=range(len(SHAPES)))
print('Confusion (rows = labeled, columns = predicted):')
print('        ' + ' '.join(f'{s[:6]:>7s}' for s in SHAPES))
for s, row in zip(SHAPES, cm):
    print(f'{s:7s} ' + ' '.join(f'{v:7d}' for v in row))

## 7. Save the model and the results
Everything goes to the run folder in Drive: the Keras model, a TensorFlow Lite version for the
browser (float16, about half the size), and a JSON summary of the numbers above.

In [ ]:
model.save(str(RUN_DIR / 'face_shape.keras'))

export_dir = '/content/saved_model'
shutil.rmtree(export_dir, ignore_errors=True)
model.export(export_dir)
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
(RUN_DIR / 'face_shape.tflite').write_bytes(converter.convert())
(RUN_DIR / 'labels.txt').write_text('\n'.join(SHAPES) + '\n')

summary = {
    'backbone': BACKBONE,
    'dataset': Path(DATA_ZIP).stem,
    'split': 'person-disjoint (StratifiedGroupKFold on average-linkage Facenet512 identity groups)',
    'same_person_similarity': SAME_PERSON_SIMILARITY,
    'people': int(len(sizes)),
    'people_with_conflicting_labels': len(mixed),
    'train_photos': int(len(train_idx)), 'val_photos': int(len(val_idx)), 'test_photos': int(len(test_idx)),
    'test_people': len(set(groups[test_idx])),
    'test_accuracy': round(accuracy, 4),
    'per_shape_recall': {s: round(float(cm[i, i] / max(cm[i].sum(), 1)), 4) for i, s in enumerate(SHAPES)},
    'confusion': cm.tolist(),
    'crop': {'size': IMG_SIZE, 'margin': CROP_MARGIN, 'levelled_by_eyes': True},
    'created': time.strftime('%Y-%m-%dT%H:%M:%S'),
}
(RUN_DIR / 'results.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('\nSaved to', RUN_DIR, [p.name for p in RUN_DIR.iterdir()])